In [ ]:
import os

import requests
import yaml

print(os.getcwd())

In [ ]:
def download_requests(url: str, output_file_path: str) -> bool:
    r = requests.get(url, stream=True, allow_redirects=True)
    if r.status_code == 200:
        # print(f"{r.url}, {r.status_code}, {r.headers}")
        with open(output_file_path, "wb") as fp:
            for chunk in r.iter_content(1024 * 1024):
                fp.write(chunk)
        # if "Content-Length" in r.headers:
        #     if int(os.path.getsize(output_file_path)) == int(r.headers["Content-Length"]):
        return True
        # else:
        #     return False
    else:
        return False

In [ ]:
with open("data/datasets.yaml", encoding="utf-8") as fp:
    datasets = yaml.safe_load(
        fp
    )  # fd.FlatDict(yaml.safe_load(fp) or {}, delimiter="/")
for mime_type, examples in datasets.items():
    if not isinstance(examples, dict):
        continue
    for example, metadata in examples.items():
        # print(f"{mime_type}, {example}, {metadata.keys()}")
        if not all(concept in metadata for concept in ("name", "spdx")):
            continue
        if "url" in metadata:  # need to download
            if metadata["url"].count(":") == 2:  # compressed
                if (
                    f"header" in metadata
                ):  # endpoint Cambridge API particularly tricky...
                    pass
                else:
                    archive_link, data_file_name = metadata["url"].rsplit(":", 1)
                    print(
                        f"remote file, compressed >>>> {archive_link}, {data_file_name}, {os.getcwd()}/{archive_link.rsplit('/')[-1]}"
                    )
                    print(f">>>> {metadata['name']}")
                    status = download_requests(
                        archive_link, f"{archive_link.rsplit('/')[-1]}"
                    )
            else:
                print(f"remote file, not compressed >>>> {metadata['url']}")
                print(f">>>> {metadata['name']}")
                # status = download_requests(metadata['url'], metadata["name"])
        else:
            if os.path.isfile(f"data/{mime_type}/{example}/{metadata['name']}"):
                print(
                    f"local file, not compressed >>>> data/{mime_type}/{example}/{metadata['name']}"
                )
    """
    if key.endswith(f"\\@origin") and f"{key.rsplit('/', 1)[0]}/\\@name":
        case = key.rsplit("/", 1)[0]
        print(case)
        if os.path.isfile(f"data/{value}"):  # local file, never compressed by default
            print(f"local file, not compressed >>>> data/{value}")
        elif value.startswith("https://"):  # file to download
            if value.count(":") == 2:  # compressed
                archive_link, data_file_name = value.rsplit(":", 1)
                print(
                    f"remote file, compressed >>>> {archive_link}, {data_file_name}, {os.getcwd()}/{archive_link.rsplit('/')[-1]}"
                )
                # status = download_requests(
                #     archive_link, f"{archive_link.rsplit('/')[-1]}"
                # )
            else:
                print(f"remote file, not compressed >>>> {value}")
                # status = download_requests(value, f"{value.rsplit('/')[-1]}")
        else:
            continue
    """